# Task 4 – Trend Analysis by Release Year
**Auspify Technologies – Data Analysis Using Python Internship**

**Goal:** Analyze how Netflix content production has changed over time.

**Workflow:** Organize content by release year → Calculate yearly releases → Identify growth and decline trends → Create trend visualizations → Interpret business patterns

## Step 1: Organize content by release year

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RED, BLACK = '#E50914', '#221F1F'
os.makedirs('charts', exist_ok=True)

path = '../Task1/netflix_cleaned.csv' if os.path.exists('../Task1/netflix_cleaned.csv') else 'netflix_cleaned.csv'
df = pd.read_csv(path, parse_dates=['date_added'])
print('Loaded:', path, '| Shape:', df.shape)
print('Release years range:', df['release_year'].min(), 'to', df['release_year'].max())
print('Missing release_year values:', df['release_year'].isnull().sum())
df[['title', 'type', 'release_year', 'year_added']].head()

In [ ]:
# Organize titles into decades and eras
df['decade'] = (df['release_year'] // 10 * 10).astype(int)
df['era'] = pd.cut(df['release_year'], bins=[1900, 1999, 2009, 2014, 2018, 2030],
                   labels=['Before 2000', '2000-2009', '2010-2014', '2015-2018', '2019-2021'])
df.groupby('era', observed=True).size().to_frame('titles')

## Step 2: Calculate yearly content releases

In [ ]:
years = range(df['release_year'].min(), df['release_year'].max() + 1)

yearly = df.groupby('release_year').size().reindex(years, fill_value=0).rename('titles').to_frame()
by_type = df.groupby(['release_year', 'type']).size().unstack(fill_value=0).reindex(years, fill_value=0)

yearly['movies'] = by_type['Movie']
yearly['tv_shows'] = by_type['TV Show']
yearly['yoy_change'] = yearly['titles'].diff()
yearly['yoy_pct'] = (yearly['titles'].pct_change().replace([np.inf, -np.inf], np.nan) * 100).round(1)
yearly['moving_avg_3yr'] = yearly['titles'].rolling(3).mean().round(1)
yearly['trend'] = np.where(yearly['yoy_change'] > 0, 'Growth', np.where(yearly['yoy_change'] < 0, 'Decline', 'Flat'))
yearly.index.name = 'release_year'

yearly.to_csv('yearly_release_summary.csv')
print('Report saved: yearly_release_summary.csv')
yearly.loc[2005:2021]

## Step 3: Identify growth and decline trends

In [ ]:
modern = yearly.loc[2000:]            # years with enough titles for reliable trends
peak_year = yearly['titles'].idxmax()
print(f"Peak release year        : {peak_year} ({yearly.loc[peak_year, 'titles']:,} titles)")
print(f"Titles released 2010+    : {df[df['release_year'] >= 2010].shape[0] / len(df) * 100:.1f}% of the catalogue")
print(f"Titles released before 2000: {df[df['release_year'] < 2000].shape[0] / len(df) * 100:.1f}% of the catalogue")

# Compound annual growth rate (CAGR) during the boom
start, end = 2010, 2018
cagr = (yearly.loc[end, 'titles'] / yearly.loc[start, 'titles']) ** (1 / (end - start)) - 1
print(f"CAGR {start}-{end}          : {cagr * 100:.1f}% per year ({yearly.loc[start, 'titles']} -> {yearly.loc[end, 'titles']:,} titles)")

# Biggest single-year growth and the decline years
growth_years = modern[modern.index <= 2018]
best = growth_years['yoy_pct'].idxmax()
print(f"Biggest yearly growth    : {best} ({growth_years.loc[best, 'yoy_pct']:+.1f}%)")
declines = modern[(modern['trend'] == 'Decline')]
print('Years with a decline since 2000:', declines.index.tolist())

In [ ]:
# Growth phases
phases = [('2000-2009', 2000, 2009), ('2010-2014', 2010, 2014), ('2015-2018', 2015, 2018), ('2019-2021', 2019, 2021)]
rows = []
for name, a, b in phases:
    sub = yearly.loc[a:b]
    rows.append({'Period': name, 'Titles': int(sub['titles'].sum()),
                 'Average per year': round(sub['titles'].mean(), 1),
                 'Share of catalogue (%)': round(sub['titles'].sum() / len(df) * 100, 1)})
phase_table = pd.DataFrame(rows).set_index('Period')
phase_table

## Step 4: Create trend visualizations
### 4.1 Titles by release year (all years)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
x, y = yearly.index.values, yearly['titles'].values
ax.fill_between(x, y, color=RED, alpha=0.18)
ax.plot(x[:-1], y[:-1], color=RED, linewidth=2.5)
ax.plot(x[-2:], y[-2:], color=RED, linewidth=2.5, linestyle='--')   # 2021 is a partial year
ax.annotate(f"Peak: {peak_year}\n{yearly.loc[peak_year, 'titles']:,} titles", (peak_year, yearly.loc[peak_year, 'titles']),
            textcoords='offset points', xytext=(-95, -5), fontweight='bold', arrowprops=dict(arrowstyle='->', color=BLACK))
ax.set_title('Netflix Titles by Release Year (1925-2021)')
ax.set_xlabel('Release year')
ax.set_ylabel('Number of titles')
plt.tight_layout()
plt.savefig('charts/01_titles_by_release_year.png', bbox_inches='tight')
plt.show()

*The dashed segment marks 2021, a partial year (the dataset ends in September 2021).*

### 4.2 Movies vs TV Shows by release year (2000 onwards)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for col, label, color in [('movies', 'Movies', RED), ('tv_shows', 'TV Shows', BLACK)]:
    s = modern[col]
    ax.plot(s.index[:-1], s.values[:-1], marker='o', linewidth=2.5, color=color, label=label)
    ax.plot(s.index[-2:], s.values[-2:], linewidth=2.5, color=color, linestyle='--')
ax.set_title('Movies vs TV Shows by Release Year (2000-2021)')
ax.set_xlabel('Release year')
ax.set_ylabel('Number of titles')
ax.set_xticks(range(2000, 2022, 2))
ax.legend()
plt.tight_layout()
plt.savefig('charts/02_movies_vs_tv_by_year.png', bbox_inches='tight')
plt.show()

tv_ahead = modern[modern['tv_shows'] > modern['movies']].index.tolist()
print('Years since 2000 when TV Show releases exceeded Movie releases:', tv_ahead)

### 4.3 Year-over-year growth (2005 onwards)

In [ ]:
yoy = yearly.loc[2005:, 'yoy_pct']

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.bar(yoy.index, yoy.values, color=[BLACK if v >= 0 else RED for v in yoy.values])
for xi, v in zip(yoy.index, yoy.values):
    ax.text(xi, v + (2 if v >= 0 else -6), f'{v:+.0f}%', ha='center', fontsize=8.5)
ax.axhline(0, color='grey', linewidth=1)
ax.set_title('Year-over-Year Change in Titles Released\nblack = growth, red = decline (2021 is a partial year)')
ax.set_xlabel('Release year')
ax.set_ylabel('Change vs previous year (%)')
ax.set_xticks(yoy.index)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('charts/03_yoy_change.png', bbox_inches='tight')
plt.show()

### 4.4 Titles by decade of release

In [ ]:
dec = df.groupby('decade').size()
dec = dec[dec.index >= 1980]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar([f'{d}s' for d in dec.index], dec.values, color=RED)
for i, v in enumerate(dec.values):
    ax.text(i, v + 80, f'{v:,}\n({v / len(df) * 100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Titles by Release Decade (1980s onwards)')
ax.set_ylabel('Number of titles')
ax.set_ylim(0, dec.max() * 1.2)
plt.tight_layout()
plt.savefig('charts/04_titles_by_decade.png', bbox_inches='tight')
plt.show()
print('Note: the 2020s bar covers only 2020 and part of 2021.')

### 4.5 How quickly does content reach Netflix? (release year vs year added)

In [ ]:
# Years between a title's release and its addition to Netflix (negative values = added before release year, set to 0)
df['years_to_netflix'] = (df['year_added'] - df['release_year']).clip(lower=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

gap = df['years_to_netflix'].clip(upper=10).value_counts(normalize=True).sort_index() * 100
labels = [str(int(i)) if i < 10 else '10+' for i in gap.index]
axes[0].bar(labels, gap.values, color=RED)
axes[0].set_title('Years Between Release and Netflix Addition')
axes[0].set_xlabel('Years')
axes[0].set_ylabel('% of titles')

fresh = (df.assign(fresh=df['years_to_netflix'] <= 1).groupby('year_added')['fresh'].mean() * 100).loc[2014:2020]
axes[1].plot(fresh.index, fresh.values, marker='o', linewidth=2.5, color=BLACK)
axes[1].set_title('Share of Titles Added Within 1 Year of Release')
axes[1].set_xlabel('Year added')
axes[1].set_ylabel('% of titles added that year')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('charts/05_release_to_netflix_gap.png', bbox_inches='tight')
plt.show()

print(f"Median gap: {df['years_to_netflix'].median():.0f} year(s)")
print(f"Added within 1 year of release: {(df['years_to_netflix'] <= 1).mean() * 100:.1f}%")
print(f"Added 10+ years after release : {(df['years_to_netflix'] >= 10).mean() * 100:.1f}%")
fresh.round(1).to_frame('% added within 1 year')

### 4.6 Combined trend dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Netflix Trend Analysis by Release Year', fontsize=18, fontweight='bold')

# (a) all titles + moving average (2000 onwards)
m = modern
axes[0, 0].plot(m.index[:-1], m['titles'].values[:-1], marker='o', color=RED, linewidth=2.5, label='Titles per year')
axes[0, 0].plot(m.index[-2:], m['titles'].values[-2:], color=RED, linewidth=2.5, linestyle='--')
axes[0, 0].plot(m.index, m['moving_avg_3yr'], color=BLACK, linewidth=2, linestyle=':', label='3-year average')
axes[0, 0].set_title('Titles Released per Year (2000-2021)')
axes[0, 0].set_xticks(range(2000, 2022, 3))
axes[0, 0].legend()

# (b) movies vs TV
for col, label, color in [('movies', 'Movies', RED), ('tv_shows', 'TV Shows', BLACK)]:
    axes[0, 1].plot(m.index[:-1], m[col].values[:-1], marker='o', linewidth=2.3, color=color, label=label)
    axes[0, 1].plot(m.index[-2:], m[col].values[-2:], linewidth=2.3, color=color, linestyle='--')
axes[0, 1].set_title('Movies vs TV Shows by Release Year')
axes[0, 1].set_xticks(range(2000, 2022, 3))
axes[0, 1].legend()

# (c) YoY change
y2 = yearly.loc[2008:, 'yoy_pct']
axes[1, 0].bar(y2.index, y2.values, color=[BLACK if v >= 0 else RED for v in y2.values])
axes[1, 0].axhline(0, color='grey', linewidth=1)
axes[1, 0].set_title('Year-over-Year Change (%)')
axes[1, 0].set_xticks(y2.index)
axes[1, 0].tick_params(axis='x', rotation=60)

# (d) decades
axes[1, 1].bar([f'{d}s' for d in dec.index], dec.values, color=RED)
for i, v in enumerate(dec.values):
    axes[1, 1].text(i, v + 80, f'{v:,}', ha='center', fontweight='bold')
axes[1, 1].set_title('Titles by Release Decade')
axes[1, 1].set_ylim(0, dec.max() * 1.15)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('charts/06_trend_dashboard.png', bbox_inches='tight')
plt.show()

## Step 5: Interpret business patterns
### Key findings (calculated from the data)

In [ ]:
movie_peak = by_type['Movie'].idxmax()
tv_peak = by_type['TV Show'].idxmax()
streak = modern.loc[:2018]
growth_run = 0
for y_ in range(2018, 1999, -1):
    if modern.loc[y_, 'yoy_change'] > 0: growth_run += 1
    else: break

print('KEY FINDINGS')
print('-' * 70)
print(f"1. Releases range from {df['release_year'].min()} to {df['release_year'].max()}, but {df[df['release_year'] >= 2010].shape[0] / len(df) * 100:.0f}% of titles were released in 2010 or later; only {df[df['release_year'] < 2000].shape[0] / len(df) * 100:.0f}% were released before 2000.")
print(f"2. Releases peaked in {peak_year} with {yearly.loc[peak_year, 'titles']:,} titles, after {growth_run} consecutive years of growth ({2018 - growth_run + 1}-2018).")
print(f"3. From 2010 to 2018, yearly releases grew from {yearly.loc[2010, 'titles']} to {yearly.loc[2018, 'titles']:,} titles, about {cagr * 100:.0f}% per year.")
print(f"4. The steepest jump was {best} ({growth_years.loc[best, 'yoy_pct']:+.0f}%), when releases rose from {yearly.loc[best - 1, 'titles']} to {yearly.loc[best, 'titles']}.")
print(f"5. Releases then declined: {yearly.loc[2019, 'titles']:,} in 2019 ({yearly.loc[2019, 'yoy_pct']:+.0f}%), {yearly.loc[2020, 'titles']:,} in 2020 ({yearly.loc[2020, 'yoy_pct']:+.0f}%) and {yearly.loc[2021, 'titles']:,} in 2021 (partial year).")
print(f"6. Movie releases peaked in {movie_peak} ({by_type['Movie'].max()} titles); TV Show releases peaked in {tv_peak} ({by_type['TV Show'].max()} titles).")
print(f"7. TV Show releases exceeded Movie releases in: {', '.join(map(str, tv_ahead))} (2021 is a partial year).")
print(f"8. The median title was added {df['years_to_netflix'].median():.0f} year after release; {(df['years_to_netflix'] <= 1).mean() * 100:.0f}% were added within a year, while {(df['years_to_netflix'] >= 10).mean() * 100:.0f}% were 10+ years old.")

### Business insights
- **The catalogue is dominated by recent releases.** 85% of titles were released in 2010 or later, and only 6% before 2000, so Netflix's library is built mainly around recent content.
- **There was a strong growth period.** Yearly releases grew for seven years in a row (2012 to 2018), from 192 to 1,144 titles, about 25% per year. The steepest jump was 2016 (+62%).
- **Growth then slowed and reversed.** Releases fell 10% in 2019 and 7% in 2020. The 2021 figure (592) is a partial year, so it should not be read as a full collapse. The data shows the pattern but not its cause.
- **The mix is shifting toward series.** Movie releases peaked in 2018 (766) and fell to 517 in 2020, while TV Show releases peaked in 2020 (436). In 2021, TV Show titles (315) exceeded movies (277) for the first time since 2000, in a partial year.
- **Content is mostly fresh, but less so over time.** The median title was added one year after release, and 55% were added within a year. However, the share added within a year of release fell from 83% in 2014 to 52% in 2019, so later library growth relied more on older titles.
- **Recommendations:**
  1. Track new releases per year as a core KPI, and re-check the trend once 2021 is complete.
  2. Keep investing in series, since TV Show releases are holding up better than movie releases.
  3. Balance new releases with older catalogue titles to keep the library both fresh and deep.

*Limitations: the data shows the release year of titles in the catalogue, not only Netflix's own productions, so it includes licensed content. The dataset ends in September 2021, and years before 1980 have very few titles.*

## Summary
| Item | Result |
|---|---|
| Dataset | `netflix_cleaned.csv` (8,786 titles, release years 1925 to 2021) |
| Report file | `yearly_release_summary.csv` (yearly counts, change, 3-year average, trend) |
| Charts saved | `Task4/charts/` (6 PNG files) |
| Skills used | Time series analysis, trend identification, growth rates, Matplotlib, Seaborn |